# Module 02 LAB B: Asynchronous Timing Trap -- Colab GPU part

**Status: NOT EXECUTED.** Authored without GPU access; no cell has run on Colab yet. Treat every
environment assumption below as `TODO_VERIFY` until a pilot run records a manifest.

- Runtime: *Runtime > Change runtime type > T4 GPU* (free tier, sm_75: no FP8, no bf16 tensor cores, no FlashAttention-2).
- Commit `report.md` sections 1-2 (prediction, design) **before** running this notebook.
- Sessions disconnect: every result is written to Google Drive, append-only, one directory per run.
- Never run this on a local GPU; `common/gpu.py` refuses outside Colab unless `LAB_ALLOW_GPU=1`.
- Runs the learner's `starter/timing.py` (or the reference) against `torch.cuda` via `experiments/torch_device.py`.
- Compare with the simulator results from `experiments/sim_timing_sweep.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/aie-labs/02/lab-b-async-timing'
import os; os.makedirs(OUT, exist_ok=True); print(OUT)

In [ ]:
# Clone (placeholder URL -- replace if the course repo moves) and locate this lab.
!git clone --depth 1 https://github.com/mrbrownnn/Advanced-AI-Engineering /content/aie 2>/dev/null || git -C /content/aie pull --ff-only
import glob, os
LAB = glob.glob('/content/aie/**/02-inference-gpu-fundamentals/labs/lab-b-async-timing', recursive=True)[0]
os.chdir(LAB); print(LAB)
!git -C /content/aie rev-parse HEAD

In [ ]:
# Pinned lab deps. torch/CUDA are Colab's preinstalled build: record, don't reinstall.
!pip -q install hypothesis==6.165.10 pyyaml==6.0.3 pytest==9.0.3
!nvidia-smi
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm,clocks.max.mem,power.limit --format=csv
import torch, numpy, sys
print(sys.version, 'torch', torch.__version__, 'cuda', torch.version.cuda, 'numpy', numpy.__version__)
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0))

In [ ]:
# Manifest to Drive (append-only).
import sys, json, datetime
sys.path.insert(0, os.path.join(LAB, '..', 'common'))
import labkit
m = labkit.manifest(seed=0, extra_packages=('torch',))
p = os.path.join(OUT, 'manifest-' + datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ') + '.json')
open(p, 'x').write(json.dumps(m, indent=1)); print(p)

In [ ]:
# CPU core first: the same tests the learner runs locally (starter by default).
# Set LAB_IMPL=solution only in the instructor copy.
!python -m pytest tests -q

In [ ]:
# Counter probe: can Nsight Compute read device counters here? (TODO_VERIFY: expected to fail with
# ERR_NVGPUCTRPERM on shared VMs; if so, the lab falls back to CUDA-event timing + analytical bytes/FLOPs.)
import gpu
status = gpu.ncu_status(); print({k: v for k, v in status.items() if k != 'tail'})

In [ ]:
# Measurement (writes raw.jsonl + result.yaml into a new directory under OUT).
!python experiments/gpu_timing.py --out "$OUT"

In [ ]:
import glob, yaml
latest = sorted(glob.glob(os.path.join(OUT, '*-timing')))[-1]
res = yaml.safe_load(open(os.path.join(latest, 'result.yaml')))['result']
print(latest)
for k, v in res['aggregated_metrics'].items():
    print(f"{k:<42} warm median {v['warm_median']*1e3:9.3f} ms  cold {v['cold']}")

Copy the result directory path and checksums into `report.md` section 4. If the counter probe
failed, say so in section 8 and keep conclusions at the useful-byte / analytical boundary.